In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
from typing import List, Dict, Tuple
import json
from collections import defaultdict
import random
import numpy as np
import os
from tqdm import tqdm
import re
import warnings
warnings.filterwarnings('ignore')
from transformers import logging
logging.set_verbosity_error()

%env CUDA_VISIBLE_DEVICES=4,5,6

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env: CUDA_VISIBLE_DEVICES=4,5,6


In [ ]:
# Загружаем MMLU

mmlu_dataset = load_dataset("cais/mmlu", "all")
test_samples_all = mmlu_dataset['test'].shuffle(seed=42).select(range(300))
print(test_samples_all)

mmlu_dataset = load_dataset("cais/mmlu", "abstract_algebra")
test_samples_alg = mmlu_dataset['test'].shuffle(seed=42).select(range(100))

mmlu_dataset = load_dataset("cais/mmlu", "college_mathematics")
test_samples_math = mmlu_dataset['test'].shuffle(seed=42).select(range(100))


def format_mmlu_question(example):
    question = example['question']
    choices = example['choices']
    return f"""The following is a multiple choice question. Choose the correct answer from the options.

Question: {question}

Options:
A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Answer with the letter of the correct choice only:"""

def format_math_question(example):
    question = example['problem']
    return f"""The following is a math problem. Please, reason step-by-step and put your solution in $\\boxed{{}}$ tag.

Problem: {question}

Solution:"""


Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 300
})
Train samples (Algebra, natural number answers): 100
Test samples (Algebra, natural number answers): 100


In [ ]:
def evaluate_sample(model, tokenizer, prompt):
    messages = [
    {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**model_inputs, max_new_tokens=10, do_sample=False)
    
    response = tokenizer.decode(outputs[0][model_inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

def extract_answer(text):
    matches = re.findall(r'\b([ABCD])\b', text)
    return matches[-1].upper() if matches else None

def test_accuracy(model, tokenizer, test_samples):
    correct = 0
    for example in tqdm(iter(test_samples)):
        prompt = format_mmlu_question(example)
        response = evaluate_sample(model, tokenizer, prompt)
        predicted = extract_answer(response)
        correct_answer = ['A', 'B', 'C', 'D'][example["answer"]]
        if predicted == correct_answer:
            correct += 1
    return correct / len(test_samples)

In [ ]:
def compute_expert_importance(model, tokenizer, dataset, num_samples=100):
    model.eval()
    num_layers = len(model.model.layers)
    samples = dataset.select(range(min(num_samples, len(dataset))))
    
    # Важность для каждого слоя и эксперта
    expert_importance = [defaultdict(float) for _ in range(num_layers)]
    
    for example in tqdm(samples, desc="Computing expert importance"):
        prompt = format_mmlu_question(example)
        correct_answer = ['A', 'B', 'C', 'D'][example["answer"]]
        
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        answer_text = f" {correct_answer}"
        answer_ids = tokenizer(answer_text, return_tensors="pt", add_special_tokens=False)['input_ids'].to(model.device)
        
        prompt_len = inputs['input_ids'].shape[1]
        full_ids = torch.cat([inputs['input_ids'], answer_ids], dim=1)

        # Хотим лосс только на токенах ответа - поэтому для промпта ставим дефольные в torch -100
        labels = torch.full_like(full_ids, -100)
        labels[:, prompt_len:] = answer_ids
        
        hidden_states_with_grad = {}
        mlp_inputs = {}
        
        # Cохраняем градиент для подсчета Loss
        def make_backward_hook(layer_idx):
            def hook(module, grad_input, grad_output):
                if grad_output[0] is not None:
                    hidden_states_with_grad[layer_idx] = grad_output[0].clone()
            return hook
        
        # Cохраняем вход в MLP для подсчета Loss
        def make_forward_hook(layer_idx):
            def hook(module, input, output):
                mlp_inputs[layer_idx] = input[0].detach().clone()
            return hook
        
        hooks = []
        for layer_idx in range(num_layers):
            backward_hook = model.model.layers[layer_idx].mlp.register_full_backward_hook(make_backward_hook(layer_idx))
            forward_hook = model.model.layers[layer_idx].mlp.register_forward_hook(make_forward_hook(layer_idx))
            hooks.append(backward_hook)
            hooks.append(forward_hook)
        
        outputs = model(input_ids=full_ids, labels=labels, output_hidden_states=True)
        loss = outputs.loss
        
        model.zero_grad()
        loss.backward()
        
        for hook in hooks:
            hook.remove()
        
        for layer_idx in range(num_layers):
            layer = model.model.layers[layer_idx]
            g = hidden_states_with_grad[layer_idx]
            hidden_flat = mlp_inputs[layer_idx]
            
            batch_size, seq_len, hidden_dim = hidden_flat.shape
            hidden_flat = hidden_flat.view(-1, hidden_dim)
            g_flat = g.view(-1, hidden_dim).to(hidden_flat.device)
            
            router_logits = layer.mlp.gate(hidden_flat)
            routing_weights = F.softmax(router_logits, dim=1, dtype=torch.float)
            top_k = layer.mlp.top_k
            weights, selected = torch.topk(routing_weights, top_k, dim=-1)
            
            if layer.mlp.norm_topk_prob:
                weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-6)
            
            weights = weights.to(hidden_flat.device)
            selected = selected.to(hidden_flat.device)
            
            # Берем только токены ответа
            answer_start = prompt_len
            answer_end = prompt_len + answer_ids.shape[1]
            
            for token_idx in range(answer_start, answer_end):
                grad_token_idx = token_idx - 1
                
                # Считаем важность для top_k экспертов - но можно попробовать для top_k*2 или для всех
                for k_idx in range(top_k):
                    expert_idx = selected[grad_token_idx, k_idx].item()
                    w_i = weights[grad_token_idx, k_idx].item()
                    
                    with torch.no_grad():
                        h_i = layer.mlp.experts[expert_idx](hidden_flat[grad_token_idx:grad_token_idx+1]).to(hidden_flat.device)
                    
                    g_token = g_flat[grad_token_idx]
                    # L_i = -g^T * (w_i * h_i)
                    importance = -torch.dot(g_token, (w_i * h_i).squeeze(0)).item()
                    expert_importance[layer_idx][expert_idx] += importance
        
        model.zero_grad()
    
    return expert_importance


def apply_expert_bias(model, expert_importance, eps=0.1):
    num_layers = len(model.model.layers)
    
    for layer_idx in range(num_layers):
        layer = model.model.layers[layer_idx]
        
        if len(expert_importance[layer_idx]) == 0:
            continue
        
        # Нормализуем важности
        importances = torch.zeros(layer.mlp.num_experts)
        for expert_idx, imp in expert_importance[layer_idx].items():
            importances[expert_idx] = imp
        
        # Нормализация (z-score)
        # Сейчас просто нормализуем весь вектор, но можно попробовать нормализовать только заполненные значения,
        # а для тех экспертов, которые не разу не попали в top_k, оставить bias = 0
        mean_imp = importances.mean()
        std_imp = importances.std()
        if std_imp > 1e-8:
            importances = (importances - mean_imp) / std_imp
        
        # Добавляем bias
        with torch.no_grad():
            if layer.mlp.gate.bias is None:
                num_experts = layer.mlp.num_experts
                layer.mlp.gate.bias = nn.Parameter(torch.zeros(num_experts, dtype=layer.mlp.gate.weight.dtype, device=layer.mlp.gate.weight.device))
            
            layer.mlp.gate.bias += (eps * importances.to(layer.mlp.gate.bias.device))
            
        print(f"Layer {layer_idx}: bias updated (top expert importance: {importances.max():.4f})")
    
    return model


In [ ]:
# norm_topk_prob=False, потому что с True модель (дефолтная, без MoE-опитизаций) сильно хуже
# работает и с 4, и с 8 экспертами

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-MoE-A2.7B-Chat")
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-MoE-A2.7B-Chat", torch_dtype=torch.float16, device_map="auto", num_experts_per_tok=8, norm_topk_prob=False)

# accuracy = test_accuracy(model, tokenizer, test_samples_math)
# print("math:", accuracy)
# accuracy = test_accuracy(model, tokenizer, test_samples_alg)
# print("alg:", accuracy)
# accuracy = test_accuracy(model, tokenizer, test_samples_all)
# print("all:", accuracy)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-MoE-A2.7B-Chat")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen1.5-MoE-A2.7B-Chat", torch_dtype=torch.float16, device_map="auto", num_experts_per_tok=8, norm_topk_prob=False)

expert_importance = compute_expert_importance(model, tokenizer, test_samples_math, num_samples=100)
model = apply_expert_bias(model, expert_importance, eps=0.1)

accuracy = test_accuracy(model, tokenizer, test_samples_math)
print("math:", accuracy)
accuracy = test_accuracy(model, tokenizer, test_samples_alg)
print("alg:", accuracy)
accuracy = test_accuracy(model, tokenizer, test_samples_all)
print("all:", accuracy)

Computing expert importance: 100%|██████████| 100/100 [10:22<00:00,  6.22s/it]


Layer 0: bias updated (top expert importance: 1.7991)
Layer 1: bias updated (top expert importance: 4.7316)
Layer 2: bias updated (top expert importance: 5.5716)
Layer 3: bias updated (top expert importance: 0.8228)
Layer 4: bias updated (top expert importance: 5.4065)
Layer 5: bias updated (top expert importance: 7.0322)
Layer 6: bias updated (top expert importance: 1.1922)
Layer 7: bias updated (top expert importance: 4.7797)
Layer 8: bias updated (top expert importance: 2.3836)
Layer 9: bias updated (top expert importance: 6.5896)
Layer 10: bias updated (top expert importance: 4.8207)
Layer 11: bias updated (top expert importance: 5.3831)
Layer 12: bias updated (top expert importance: 5.8380)
Layer 13: bias updated (top expert importance: 5.0041)
Layer 14: bias updated (top expert importance: 3.5389)
Layer 15: bias updated (top expert importance: 1.9388)
Layer 16: bias updated (top expert importance: 3.9889)
Layer 17: bias updated (top expert importance: 1.4326)
Layer 18: bias updat

100it [07:44,  4.64s/it]


math: 0.36


100it [07:48,  4.68s/it]


alg: 0.35


300it [25:06,  5.02s/it]

all: 0.5533333333333333
